In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [1]:
pass  # Colab only: from google.colab import drive
pass  # Colab only: drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

SAVE_DIR = DATA_DIR + "/datasets/"

df = pd.read_csv(SAVE_DIR + "collisions_dataset_clean.csv",
                 parse_dates=['datetime'],
                 dtype={'collision_index': str},
                 low_memory=False)

print(f"Loaded: {len(df):,} rows")

Loaded: 503,407 rows


In [3]:
# 1. Fixing the -1 speed limit (these are likely motorways recorded incorrectly)
df['speed_limit'] = df['speed_limit'].replace(-1, 70)

In [4]:
# 2. Time-of-day risk buckets (more granular than binary rush hour)
def time_bucket(hour):
    if 0 <= hour < 6:   return 'late_night'      # high severity, low volume
    if 6 <= hour < 9:   return 'morning_rush'
    if 9 <= hour < 12:  return 'mid_morning'
    if 12 <= hour < 14: return 'lunch'
    if 14 <= hour < 17: return 'afternoon'
    if 17 <= hour < 20: return 'evening_rush'     # highest volume
    if 20 <= hour < 24: return 'night'            # higher severity

df['time_bucket'] = df['hour'].apply(time_bucket)

In [5]:
# 3. Darkness flag (cleaner than raw light_conditions)
df['is_dark'] = (df['light_conditions'] != 1).astype(int)
df['is_dark_no_lights'] = (df['light_conditions'] == 6).astype(int)

In [6]:
# 4. Adverse conditions flags
df['is_wet_road']    = df['road_surface_conditions'].isin([2]).astype(int)
df['is_icy_road']    = df['road_surface_conditions'].isin([4, 3]).astype(int)
df['is_bad_weather'] = df['weather_conditions'].isin([2, 3, 5, 6, 7]).astype(int)


In [7]:
# 5. High speed flag
df['is_high_speed'] = (df['speed_limit'] >= 60).astype(int)
df['is_rural_road']  = (df['urban_or_rural_area'] == 2).astype(int)

In [8]:
# 6. Junction flag
# junction_detail: 0 = not at junction, anything else = at a junction
df['at_junction'] = (df['junction_detail'] != 0).astype(int)


In [9]:
# 7. Key interaction features (these capture compounding risk)
df['dark_and_wet']        = df['is_dark'] * df['is_wet_road']
df['dark_and_no_lights']  = df['is_dark_no_lights'] * df['is_rural_road']
df['high_speed_and_dark'] = df['is_high_speed'] * df['is_dark']
df['high_speed_and_wet']  = df['is_high_speed'] * df['is_wet_road']
df['rural_and_dark']      = df['is_rural_road'] * df['is_dark']
df['friday_evening']      = ((df['day_of_week'] == 4) &
                              (df['time_bucket'] == 'evening_rush')).astype(int)

In [10]:
# 8. Dual carriageway flag (deadliest road type from EDA)
df['is_dual_carriageway'] = (df['road_type'] == 3).astype(int)

In [11]:
# 9. Season
def get_season(month):
    if month in [12, 1, 2]: return 'winter'
    if month in [3, 4, 5]:  return 'spring'
    if month in [6, 7, 8]:  return 'summer'
    return 'autumn'

df['season'] = df['month'].apply(get_season)

In [12]:
# 10. Binary severity target (for classification)
# We'll predict: is this accident serious or fatal? (vs slight)
# For the risk model: 1 = serious/fatal, 0 = slight
df['is_serious'] = (df['collision_severity'] <= 2).astype(int)

In [14]:
# Verify New features
new_features = ['time_bucket', 'is_dark', 'is_dark_no_lights', 'is_wet_road',
                'is_icy_road', 'is_bad_weather', 'is_high_speed', 'is_rural_road',
                'at_junction', 'dark_and_wet', 'high_speed_and_dark',
                'high_speed_and_wet', 'rural_and_dark', 'friday_evening',
                'is_dual_carriageway', 'season', 'is_serious']

print("\n New features created ")
for f in new_features:
    if df[f].dtype == object:
        print(f"  {f}: {df[f].value_counts().to_dict()}")
    else:
        print(f"  {f}: mean={df[f].mean():.3f}, sum={df[f].sum():,}")


 New features created 
  time_bucket: {'afternoon': 115509, 'evening_rush': 106811, 'mid_morning': 69975, 'night': 62125, 'morning_rush': 61039, 'lunch': 59657, 'late_night': 28291}
  is_dark: mean=0.285, sum=143,620
  is_dark_no_lights: mean=0.053, sum=26,587
  is_wet_road: mean=0.247, sum=124,484
  is_icy_road: mean=0.015, sum=7,430
  is_bad_weather: mean=0.134, sum=67,464
  is_high_speed: mean=0.176, sum=88,743
  is_rural_road: mean=0.325, sum=163,504
  at_junction: mean=0.495, sum=248,994
  dark_and_wet: mean=0.112, sum=56,342
  high_speed_and_dark: mean=0.051, sum=25,816
  high_speed_and_wet: mean=0.055, sum=27,848
  rural_and_dark: mean=0.088, sum=44,445
  friday_evening: mean=0.036, sum=17,943
  is_dual_carriageway: mean=0.148, sum=74,703
  season: {'autumn': 137121, 'summer': 132237, 'winter': 118823, 'spring': 115226}
  is_serious: mean=0.233, sum=117,450


In [15]:
# Quick check: do our interaction features correlate with serious outcomes?
print("\n Interaction feature --> serious accident rate")
for feat in ['is_dark', 'is_dark_no_lights', 'is_wet_road', 'is_icy_road',
             'is_high_speed', 'at_junction', 'dark_and_wet',
             'high_speed_and_dark', 'high_speed_and_wet', 'rural_and_dark']:
    rate_when_1 = df[df[feat]==1]['is_serious'].mean()
    rate_when_0 = df[df[feat]==0]['is_serious'].mean()
    uplift = rate_when_1 / rate_when_0
    print(f"  {feat:30s} --> serious rate: {rate_when_1:.3f} vs {rate_when_0:.3f}  (uplift: {uplift:.2f}x)")


 Interaction feature --> serious accident rate
  is_dark                        --> serious rate: 0.258 vs 0.223  (uplift: 1.15x)
  is_dark_no_lights              --> serious rate: 0.342 vs 0.227  (uplift: 1.51x)
  is_wet_road                    --> serious rate: 0.243 vs 0.230  (uplift: 1.05x)
  is_icy_road                    --> serious rate: 0.222 vs 0.233  (uplift: 0.95x)
  is_high_speed                  --> serious rate: 0.309 vs 0.217  (uplift: 1.42x)
  at_junction                    --> serious rate: 0.218 vs 0.249  (uplift: 0.88x)
  dark_and_wet                   --> serious rate: 0.269 vs 0.229  (uplift: 1.17x)
  high_speed_and_dark            --> serious rate: 0.322 vs 0.228  (uplift: 1.41x)
  high_speed_and_wet             --> serious rate: 0.291 vs 0.230  (uplift: 1.26x)
  rural_and_dark                 --> serious rate: 0.311 vs 0.226  (uplift: 1.38x)


In [16]:
# Save
df.to_csv(SAVE_DIR + "collisions_features.csv", index=False)